# Подготовка данных и моделирование

Кратко: ноутбук содержит очистку, фичеризацию и базовые модели для дневной капитализации.


In [ ]:
from config import catalog
from config.columns import dtype_dict_raw
import pandas as pd
# import numpy as np
from dotenv import load_dotenv


load_dotenv(config.PROJ_ROOT)

y_name = 'dailycapitalization'
secid = ['secid']
tradedate = ['tradedate']
index = secid + tradedate
pd.set_option("display.max_columns", None)

## Подготовка данных


Чтение данных

In [ ]:
df = pd.read_csv(
    config.DATA_DIR / 'raw' / 'dataset.csv',
    parse_dates=['tradedate'],
    sep='\t',
    dtype=dtype_dict_raw,
)
df['tradedate'] = pd.to_datetime(df['tradedate'])
df['tradeyear'] = df['tradedate'].dt.year
df.shape

Анализ режимов торгов (boardid)

In [ ]:
df['boardid'].unique()

	•	TQBR — акции, основной стакан (T+2)
	•	TQBS — акции, T+ режим (вариация расчётов, реже используется)
	•	TQPI — режим для квалифицированных инвесторов
	•	TQDE — депозитарные расписки
	•	TQDP — DR / спецрежим
	•	TQNE — адресные сделки в T+
	•	TQLV — низколиквидные бумаги
	•	TQNL — неликвид / ограниченный режим
	•	EQBR — акции (аналог TQBR, но менее ликвидный)
	•	EQNE — адресные сделки
	•	EQBS — режим расчётов
	•	EQLI — листинг / малоликвидные
	•	EQNL — неликвид
	•	EQLV — low volume
	•	EQDE — депозитарные расписки
	•	EQDP — DR / спецрежим
	•	EQCC — клиринговый / спецрежим
	•	SMAL — неполные лоты (odd lots)
	•	SPEQ — negotiated / специальные сделки

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

def plot_board_volume_heatmap(df, title):
    pivot = df.pivot_table(index='tradedate', columns='boardid', values='volume', aggfunc='sum')
    sns.heatmap(pivot, cmap="viridis", norm=LogNorm())
    plt.title(title)
    plt.tight_layout()
    plt.show()

plot_board_volume_heatmap(df, "Объем торгов по режимам")

### Предобработка (sklearn pipeline)

In [ ]:
from src.steps.pipelines import build_full_pipeline

pipe = build_full_pipeline()
df = pipe.fit_transform(df)
df.shape

Save processed data

In [26]:
df.to_csv(
    config.DATA_DIR / 'processed' / 'dataset.csv',
    sep='\t',
)